# 08 — Objedinjeni zaključak projekta

Ova sveska predstavlja **završnu sintezu** celog projekta *Telco Customer
Churn*. Dok su prethodne sveske (`00`-`07`) svaka pokrivale po jedan korak
analize sa sopstvenim, lokalnim zaključkom, ovde povezujemo sve nalaze u
jedinstvenu celinu: šta smo saznali o korisnicima koji napuštaju kompaniju,
koji je model najbolji izbor i zašto, i šta bi kompanija konkretno trebalo
da uradi na osnovu ovih rezultata.

## 1. Tok projekta — pregled

| Sveska | Sadržaj |
|---|---|
| `00` | Definisanje problema, konteksta i cilja (binarna klasifikacija) |
| `01` | Inicijalni pregled — dimenzije, tipovi, skrivene nedostajuće vrednosti u `TotalCharges` |
| `02` | Provera sistematskih/logičkih grešaka između povezanih kolona |
| `03` | Stratifikovana podela na trening (80%) i test (20%) skup |
| `04` | Detekcija statističkih anomalija (IQR) — skup je statistički čist |
| `05` | Feature engineering (`ActiveServices`, `TenureGroup`) i enkodiranje |
| `06` | Iterativna EDA — statistička potvrda ključnih prediktora |
| `07` | Modelovanje, cross-validacija, tuning hiperparametara, ROC krive |
| `08` | *(ova sveska)* — sinteza i poslovne preporuke |

Kroz ceo proces dosledno smo poštovali princip da se **sve odluke o
čišćenju, transformaciji i feature engineering-u donose isključivo na
trening skupu**, dok test skup ostaje netaknut do finalne evaluacije —
čime je izbegnut *data leakage*.

## 2. Ključni nalazi o korisnicima koji napuštaju kompaniju

Na osnovu statističke analize (sveska `06`) i analize važnosti atributa
(sveska `07`), izdvajaju se sledeći obrasci — svi statistički potvrđeni
(p < 0.05) i konzistentno prepoznati kod sva tri modela:

**Tip ugovora je najsnažniji pojedinačni faktor.** Korisnici sa
`Month-to-month` ugovorom napuštaju kompaniju u ~43% slučajeva, naspram
svega ~3% kod dvogodišnjih ugovora. Kod XGBoost modela, `Contract` nosi
čak ~30% ukupne "važnosti" među svim atributima — dalеko ispred svih
ostalih.

**Dužina pretplate (`tenure`) je snažno povezana sa odlaskom.** Korisnici
koji odlaze imaju u proseku znatno kraći staž. Ovo je logično — rizik od
odlaska je najveći u prvim mesecima, pre nego što se korisnik "navikne"
na uslugu.

**Tip internet usluge igra veću ulogu nego što bi se očekivalo.**
Korisnici sa `Fiber optic` internetom imaju paradoksalno viši churn
(~42%) od korisnika sa `DSL` (~19%) — mogući uzrok je viša cena ili
razlike u kvalitetu usluge/konkurenciji na tom segmentu tržišta; ovo je
korelacija koju bi kompanija trebalo dodatno da istraži.

**Način plaćanja ukazuje na manje angažovane korisnike.** `Electronic
check` korisnici (najmanje automatizovan način plaćanja) napuštaju
znatno češće (~45%) od korisnika sa automatskim plaćanjem (~15-19%).

**Nedostatak dodatnih usluga (posebno `TechSupport` i `OnlineSecurity`)
povećava rizik od odlaska.** Ovo je i motivacija iza novog feature-a
`ActiveServices`, koji se pokazao kao koristan prediktor kod modela
baziranih na stablima.

**Napomena o uzročnosti:** navedeni nalazi su **korelacije** potvrđene
statističkim testovima i modelima, ne dokazana uzročno-posledična veza.
Na primer, ne znamo da li duži ugovor *uzrokuje* lojalnost, ili lojalniji
korisnici jednostavno biraju duže ugovore. Za pravu uzročnost bio bi
potreban kontrolisan eksperiment (npr. A/B test ponude dužih ugovora).

In [1]:
import pandas as pd

# Rezime finalnih rezultata sva tri modela (pre podešavanja hiperparametara)
rezime_finalni = pd.DataFrame({
    "Model": ["Logistička regresija", "Random Forest", "XGBoost"],
    "Accuracy": [0.7242, 0.7839, 0.7377],
    "Precision": [0.4884, 0.6250, 0.5043],
    "Recall": [0.7888, 0.4679, 0.7754],
    "F1-score": [0.6033, 0.5352, 0.6112],
    "ROC-AUC": [0.8342, 0.8214, 0.8314]
})
print("Rezultati na test skupu (originalni modeli):")
display(rezime_finalni)

rezime_tuning = pd.DataFrame({
    "Model": ["Logistička regresija (posle tuning-a)", "XGBoost (posle tuning-a)"],
    "Accuracy": [0.7249, 0.7306],
    "Precision": [0.4892, 0.4959],
    "Recall": [0.7861, 0.8021],
    "F1-score": [0.6031, 0.6129],
    "ROC-AUC": [0.8347, 0.8417]
})
print("\nRezultati posle podešavanja hiperparametara:")
display(rezime_tuning)

Rezultati na test skupu (originalni modeli):


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistička regresija,0.7242,0.4884,0.7888,0.6033,0.8342
1,Random Forest,0.7839,0.6250,0.4679,0.5352,0.8214
2,XGBoost,0.7377,0.5043,0.7754,0.6112,0.8314



Rezultati posle podešavanja hiperparametara:


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistička regresija (posle tuning-a),0.7249,0.4892,0.7861,0.6031,0.8347
1,XGBoost (posle tuning-a),0.7306,0.4959,0.8021,0.6129,0.8417


## 3. Koji model preporučujemo — finalna odluka

Nijedan model ne dominira u svim metrikama, pa je izbor **zavistan od
poslovnog prioriteta**:

- **Random Forest** ima najveću Accuracy (78,4%) i Precision (62,5%), ali
  najniži Recall (46,8%) — propušta više od polovine korisnika koji
  zaista odlaze. Cross-validacija (sveska `07`) je pokazala da je ovaj
  rezultat čak i optimističan — prosečan CV recall je svega ~49,8%.
- **Logistička regresija** ima najviši Recall (78,9%) i ROC-AUC (83,4%),
  uz najjednostavniju i najinterpretabilniju strukturu (koeficijenti se
  direktno čitaju).
- **XGBoost** je najbolje balansiran (najviši F1-score), a nakon
  podešavanja hiperparametara dostiže i najviši ROC-AUC (84,2%) uz
  Recall od 80,2% — najbolji sveukupni rezultat nakon tuning-a.

**Preporuka:** Za problem predviđanja churn-a, gde je poslovno mnogo
skuplje propustiti korisnika koji odlazi (False Negative) nego pogrešno
uzbuniti se za lojalnog korisnika (False Positive), prioritet je visok
**Recall**. Na osnovu toga, preporučujemo **XGBoost sa podešenim
hiperparametrima** kao finalni model — ima najbolju kombinaciju Recall-a
(80,2%) i ROC-AUC-a (84,2%) od svih testiranih varijanti. **Logistička
regresija** ostaje snažna alternativa tamo gde je interpretabilnost
prioritet (npr. objašnjavanje odluka korisničkoj podršci ili regulatoru).

## 4. Vrednost feature engineering-a

Nove promenljive kreirane u svesci `05` opravdale su svoje postojanje:

- **`ActiveServices`** se pojavljuje među atributima sa umerenim do
  visokim uticajem kod Random Forest i XGBoost modela — potvrđuje da je
  agregiranje šest kolona dodatnih usluga u jedan broj korisna
  transformacija, ne samo kozmetička.
- **`TenureGroup`** ima manji, ali ipak prisutan značaj kod modela
  baziranih na stablima (~5-6% važnosti kod oba) — modeli sami pronalaze
  slične pragove i iz sirovog `tenure`, ali grupisana verzija dodatno
  pomaže linearnom modelu (Logistička regresija) da uhvati nelinearne
  efekte.

## 5. Ograničenja rada i predlozi za dalje unapređenje

Iskreno navodimo granice ovog rada — što je i sâmo po sebi deo dobre
analitičke prakse:

- **Umerene apsolutne performanse.** F1-score od ~0,61 nije loš rezultat
  za dataset ove veličine i nebalansiranosti, ali ostavlja prostor za
  poboljšanje. Mogući sledeći koraci: probati SMOTE za sintetičko
  balansiranje klasa (umesto samo `class_weight`), ili šira pretraga
  hiperparametara (veći `n_iter` kod RandomizedSearchCV).
- **Korelacija, ne uzročnost.** Kako je navedeno u sekciji 2, nalazi
  pokazuju povezanost, ne dokazan uzrok — za poslovne odluke velikog
  obima preporučljivo je potvrditi nalaze kontrolisanim eksperimentom.
- **Multikolinearnost.** `tenure` i `TotalCharges` su jako korelisani
  (sveska `06`), što delimično "razvodnjava" njihov pojedinačni značaj
  kod Logističke regresije — model bi mogao dodatno da se pojednostavi
  izbacivanjem jedne od te dve kolone.
- **Nema praćenja modela u produkciji.** Ovaj rad se zaustavlja na
  evaluaciji na statičnom test skupu. Realna primena bi zahtevala
  periodično ponovno treniranje i praćenje pada performansi (*model
  drift*) kako se ponašanje korisnika menja tokom vremena.

## 6. Poslovne preporuke

Na osnovu svih nalaza, konkretne preporuke za telekomunikacionu
kompaniju:

1. **Prioritetno targetirati korisnike sa Month-to-month ugovorom** —
   najveći pojedinačni faktor rizika. Ponuda popusta za prelazak na
   godišnji/dvogodišnji ugovor mogla bi značajno smanjiti odliv.
2. **Posebna pažnja u prvih 6-12 meseci pretplate** — period najvišeg
   rizika od odlaska; ovde uvesti proaktivnu podršku ili uvodne popuste.
3. **Podsticati korišćenje dodatnih usluga** (`TechSupport`,
   `OnlineSecurity`) — korisnici koji ih koriste su lojalniji, verovatno
   jer je prelazak na drugog provajdera "skuplji" (izgubili bi više
   funkcionalnosti).
4. **Istražiti uzrok visokog churn-a kod Fiber optic korisnika** — nije
   jasno da li je uzrok cena, kvalitet usluge ili konkurencija; vredi
   dodatne analize pre donošenja mera.
5. **Podsticati prelazak na automatsko plaćanje** — korisnici sa
   `Electronic check` odlaze znatno češće; olakšati prelazak na
   automatsko plaćanje karticom ili preko banke.

## 7. Zaključna reč

Projekat je prošao kompletan tok analize podataka — od sirovog CSV
fajla, preko čišćenja, statistički potvrđene eksplorativne analize,
feature engineering-a, do tri istrenirana i validirana modela mašinskog
učenja sa sistematski podešenim hiperparametrima. Rezultati pokazuju da
je churn kod ovog skupa podataka predvidiv u razumnoj meri (ROC-AUC
~0,84), sa jasno identifikovanim faktorima rizika koji imaju smisla i sa
poslovne strane, ne samo statističke.

Kod, dataset i sve sveske dostupni su u GitHub repozitorijumu tima.